In [ ]:
import torch
from transformers import AutoTokenizer
from arabert.preprocess import ArabertPreprocessor
# for text classification
from transformers import AutoModelForSequenceClassification

## araBERT Preprocessor

In [ ]:
model_name="bert-base-arabert"
arabert_prep = ArabertPreprocessor(model_name=model_name)

## Initializing Model

In [ ]:
checkpoint = "aubmindlab/bert-base-arabertv02-twitter"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, return_dict=True)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

## Inference

In [ ]:
def classify(text):
    text = arabert_prep.preprocess(text)
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs)
    probs = outputs.logits.softmax(dim=1)
    return probs

In [ ]:
text = 'هذا مثال عربي اصطناعي لاختبار معالجة النصوص.'
probs = classify(text)
print(probs)

In [ ]:
# Function to process the text input
def process_text_input(text_input):
    # Here, text_input is expected to be a list of strings
    # Perform the same processing as you did with the CSV
    text_data = text_input
    # ... (rest of the processing code)

# Example usage
text_input = [
    'هذا مثال عربي اصطناعي لاختبار معالجة النصوص.',
    "0",
    # Add more text samples as needed
]

process_text_input(text_input)


## Data testing

In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from arabert.preprocess import ArabertPreprocessor
import ipywidgets as widgets

# Initialize the model and tokenizer
model_name = "bert-base-arabert"
arabert_prep = ArabertPreprocessor(model_name=model_name)
checkpoint = "aubmindlab/bert-base-arabertv02-twitter"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, return_dict=True)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# Create a text area widget
text_area = widgets.Textarea(
    value='',
    placeholder='Type something',
    description='Input:',
    disabled=False,
    layout=widgets.Layout(width='100%', height='200px')
)

# Display the text area widget
display(text_area)

# Function to classify the text
def classify(text):
    text = arabert_prep.preprocess(text)
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs)
    probs = outputs.logits.softmax(dim=1)
    return probs

# Button to trigger classification
button = widgets.Button(description="Classify")

# Output widget to display the result
output = widgets.Output()

def on_button_clicked(b):
    with output:
        output.clear_output()
        text = text_area.value
        if text.strip():
            probs = classify(text)
            print(f"Classification Probabilities: {probs}")
        else:
            print("Please enter some text to classify.")

button.on_click(on_button_clicked)

# Display the button and output
display(button, output)


In [ ]:
from transformers import AutoTokenizer
model_path = "araBERT-base_exp5/checkpoint-9351"
tokenizer = AutoTokenizer.from_pretrained(model_path)

text = dataset["train"][0]["text"]

inputs = tokenizer(text, return_tensors="pt")

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
inputs.to(device)

with torch.no_grad():
    logits = model(**inputs).logits

In [ ]:
predicted_class_id = logits.argmax().item()
model.config.id2label[predicted_class_id]

### Evaluation

In [ ]:
model_name = "bert-base-arabert"
from arabert.preprocess import ArabertPreprocessor
arabic_prep = ArabertPreprocessor(model_name=model_name)
def arabic_preprocess(examples):
    return {"text":arabert_prep.preprocess(examples["text"]), "label":examples["label"]}

In [ ]:
from tqdm.notebook import tqdm
import evaluate
import numpy as np
from sklearn.metrics import recall_score, precision_score, f1_score
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
metric = evaluate.load("glue", "sst2", device=device)
# import precision_recall_fscore_support
def evaluate_test_set(model, tokenizer, test_dataset, batch_size=32, pbar = False):
    if torch.cuda.is_available():
        device = "cuda"
    else :
        device = "cpu"
    print(f"Device: {device}")
    model.to(device)
    model.eval()
    acc = 0
    f1 = 0
    recall = 0
    precision = 0

    predictions = []
    labels = []

    test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size)
    if pbar:
        progress = tqdm(enumerate(test_dataloader), total=len(test_dataloader))
    for idx, batch in enumerate(test_dataloader):
        inputs = tokenizer(batch["text"], return_tensors="pt", padding=True, truncation=True, max_length=512)
        inputs = {name: tensor.to(device) for name, tensor in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        scores = outputs.logits
        probs = scores.softmax(dim=1)
        predicted_class_ids = probs.argmax(dim=1).tolist()

        predictions.extend(predicted_class_ids)
        batch_labels = batch["label"]
        labels.extend([x.item() for x in batch_labels])
        unique_labels = np.unique(predicted_class_ids)

        # batch_accuracy = sum([1 if label == prediction else 0 for label, prediction in zip(batch_labels, predicted_class_ids)])
        # batch_accuracy = batch_accuracy / len(batch_labels)

        batch_metric = metric.compute(predictions=predicted_class_ids, references=batch_labels)
        batch_acc = batch_metric["accuracy"]
        batch_f1 = f1_score(batch_labels, predicted_class_ids, average='weighted', labels=unique_labels)
        batch_recall = recall_score(batch_labels, predicted_class_ids, average='weighted', labels=unique_labels)
        batch_precision = precision_score(batch_labels, predicted_class_ids, average='weighted', labels=unique_labels)
        # print(batch_accuracy, batch_f1, batch_recall, batch_precision)
    
        acc += batch_metric["accuracy"]
        f1 += batch_f1
        recall += batch_recall
        precision += batch_precision

        if pbar:
            progress.update(1)
            progress.set_description(f"Batch {idx+1}/{len(test_dataloader)}: {batch_acc*100:.2f}%")
        # print(f"Batch {idx+1}/{len(test_dataloader)}: {batch_accuracy}")


    acc = acc / len(test_dataloader)
    f1 = f1 / len(test_dataloader)
    recall = recall / len(test_dataloader)
    precision = precision / len(test_dataloader)
    return acc, f1, recall, precision, predictions, labels


In [ ]:
print(f"Accuracy: {accuracy*100:.2f}%, F1: {f1*100:.2f}%, Recall: {recall*100:.2f}%, Precision: {precision*100:.2f}%")

In [ ]:
len(predictions), len(labels)
print(set(predictions))
print(set(labels))

In [ ]:
from sklearn.metrics import classification_report
cr = classification_report(labels, predictions, labels=[0, 1], target_names=["AI", "Human"])
print(cr)